# Zakuro — mesh adaptation tour

Walks through every feature of `zk.AdaptiveCompute` end-to-end:

1. **Warmup** — probe workers once, seed priors, eject unreachable, auto-set `backpressure_threshold`.
2. **Live dispatch** — watch the per-worker EMA evolve during real traffic.
3. **Soft vs greedy routing** — how `softmax_temperature` keeps the pool utilised.
4. **Node lifecycle** — `add_worker` / `remove_worker` at runtime.
5. **Health probes** — kill a worker, watch the background heartbeat mark it suspended.
6. **Drift detection** — inject a slowdown, observe the drift factor engage, then recover.

Every number is observed on spawned `zakuro-worker` subprocesses; nothing is simulated.
Runs in ~30–45 s end-to-end on a modern Mac.

In [ ]:
import signal
import time

import zakuro as zk

print("zakuro:", zk.__version__)

## 1. Spin up workers + build `AdaptiveCompute`

`zk.Worker.spawn()` runs the CLI as a subprocess and waits for `/health` to come up. The pool below has three workers on different loopback ports; `AdaptiveCompute` then wraps them with Adam-style tracking.

In [ ]:
workers = [zk.Worker.spawn(name=f"w{i}") for i in range(3)]

adaptive = zk.AdaptiveCompute(
    workers=[w.compute(verify=False) for w in workers],
    beta1=0.8,
    beta_slow=0.97,
    softmax_temperature=0.02,   # some exploration so all workers see traffic
    drift_threshold=1.5,
)

for w in workers:
    print(f"  {w}")
print(adaptive)

## 2. Warmup — calibrate priors

`warmup()` dispatches a trivial identity probe a few times per worker, seeds each worker's EMA with the observed mean, and sets `backpressure_threshold = 1.5 × max(p95)`. Without this the allocator bootstraps from the pessimistic `initial_latency` default and the first traffic burst lands on a single arbitrary worker.

In [ ]:
report = adaptive.warmup(rounds=3, timeout=5.0, verbose=True)
print(f"\nadaptive.backpressure_threshold after warmup: {adaptive.backpressure_threshold:.3f}s")
print(f"ejected workers: {report['ejected']}")
for r in report["workers"]:
    print(f"  {r.get('uri', r['idx'])}: ok={r['ok']}  p95={r.get('latency_p95', 0)*1000:.1f}ms")

## 3. Dispatch traffic — watch EMAs evolve

Define a trivial `@zk.fn`, send a batch through the allocator, and inspect `stats()` afterwards.

In [ ]:
@zk.fn
def tick(x: int) -> int:
    return x + 1

for i in range(100):
    tick.to(adaptive)(i)

print("per-worker stats after 100 dispatches:")
for i, s in enumerate(adaptive.stats()):
    fast = s["latency_ema"] * 1000
    slow = s["latency_baseline"] * 1000
    print(
        f"  worker {i}: step={s['step']:>3}  "
        f"fast_ema={fast:6.2f}ms  baseline={slow:6.2f}ms  "
        f"drift_factor={s['drift_factor']}"
    )

## 4. Greedy vs softmax

With `softmax_temperature=0` the picker is argmin — first-ms-faster wins 100 % of the traffic. With `τ > 0` the pool stays utilised and can re-balance when conditions change. Showing both on the same warmed-up pool.

In [ ]:
from collections import Counter

# greedy run (τ=0)
adaptive._tau = 0.0
picks_greedy = Counter(adaptive.pick() for _ in range(500))

# soft run (τ=0.02)
adaptive._tau = 0.02
picks_soft = Counter(adaptive.pick() for _ in range(500))

print("greedy picks over 500 calls:", dict(picks_greedy))
print("softmax picks over 500 calls:", dict(picks_soft))

## 5. Backpressure signal

`is_backpressured()` returns True when *every* worker's expected time-to-serve is above the threshold. Downstream code (`SakuraHFCallback`, custom dispatchers) can use this to skip an operation instead of piling on a saturated pool.

In [ ]:
print("current backpressure_threshold:", adaptive.backpressure_threshold, "s")
print("is_backpressured now?", adaptive.is_backpressured())

# Simulate backlog: bump every worker's queue up.
for s in adaptive._stats:
    s.queue += 20
print("is_backpressured with queues stuffed?", adaptive.is_backpressured())

# Reset for the rest of the demo.
for s in adaptive._stats:
    s.queue = 0

## 6. Node lifecycle — `add_worker` / `remove_worker`

Remove worker 0; traffic reroutes. Readmit it; the bootstrap prior (mesh-median latency) lets it earn traffic back within one batch.

In [ ]:
dropped = adaptive.remove_worker(0)
print(f"removed: {dropped.uri}")
print("remaining:", [w.uri for w in adaptive.workers])

# Dispatch 50 on the reduced pool.
for i in range(50):
    tick.to(adaptive)(i)
post_remove = [s["step"] for s in adaptive.stats()]
print("steps after 50 dispatches:", post_remove)

# Readmit the original worker.
new_idx = adaptive.add_worker(workers[0].compute(verify=False))
seeded = adaptive.stats()[new_idx]
print(
    f"readmitted at idx {new_idx}, bootstrap ema={seeded['latency_ema']*1000:.1f}ms"
)

## 7. Health probes — kill a worker, watch it suspend

`start_health_probes` spawns a daemon thread that polls each worker's `/health` (HTTP) or the QUIC `HEALTH` opcode on an interval. After `max_strikes` consecutive misses the worker is **suspended** — the picker sends its expected time to ∞, routing around it. A successful probe un-suspends.

In [ ]:
adaptive.start_health_probes(interval=0.1, timeout=0.2, max_strikes=2)

# Let baseline probes run for a moment so everyone starts healthy.
time.sleep(0.5)
print("before kill:")
for i, s in enumerate(adaptive.stats()):
    print(f"  worker {i}: suspended={s['suspended']}  strikes={s['health_strikes']}")

# SIGKILL worker 1 and watch the health probes catch on.
t_kill = time.perf_counter()
workers[1]._proc.send_signal(signal.SIGKILL)

detected_at = None
deadline = time.perf_counter() + 4.0
while time.perf_counter() < deadline:
    stats = adaptive.stats()
    # AdaptiveCompute.workers order mirrors its _stats; find the slot that
    # still points at the killed worker.
    for i, s in enumerate(stats):
        if s["suspended"]:
            detected_at = time.perf_counter() - t_kill
            print(f"\n  worker {i} suspended after {detected_at*1000:.0f} ms")
            break
    if detected_at is not None:
        break
    time.sleep(0.05)

if detected_at is None:
    print("  (worker not suspended within 4 s — unusual; try rerunning)")

print("\nstats after kill:")
for i, s in enumerate(adaptive.stats()):
    print(f"  worker {i}: suspended={s['suspended']}  strikes={s['health_strikes']}")

## 8. Drift detection — soft-demote a slow worker

Rather than killing a worker, simulate it *slowing down*: one of the workers starts serving requests that artificially wait 250 ms. The allocator's fast EMA rises above the slow baseline past `drift_threshold × baseline` → `drift_factor` jumps to the `drift_penalty` (default 5×) and the picker deflects. When the injection stops, health probes feed fast latencies back into the EMA until the ratio falls below `drift_recovery_threshold` and `drift_factor` resets to 1.0.

In [ ]:
# Slow-path function: any dispatch that lands on the chosen worker idx
# will do 250 ms of useless work before returning.
import time as _time

@zk.fn
def slow_work() -> int:
    _time.sleep(0.25)
    return 0

@zk.fn
def fast_work() -> int:
    return 0

# Which worker are we targeting for "injection"?  Pick the first healthy
# one so we can see drift kick in while other workers absorb load.
target = next(i for i, s in enumerate(adaptive.stats()) if not s["suspended"])
print(f"targeting worker {target} for slowdown injection")

# Run 5 s of traffic: whenever the allocator picks the target, route a slow
# task there; otherwise fast. This simulates 'that worker itself is slower'.
t0 = time.perf_counter()
first_drift_t = None
while time.perf_counter() - t0 < 5.0:
    idx = adaptive.pick()
    (slow_work if idx == target else fast_work).to(adaptive)()
    if first_drift_t is None:
        s = adaptive.stats()[target]
        if s["drift_factor"] > 1.0:
            first_drift_t = time.perf_counter() - t0

print(f"drift detected at t+{first_drift_t:.2f}s" if first_drift_t else "no drift observed")
for i, s in enumerate(adaptive.stats()):
    fast = s["latency_ema"] * 1000
    slow = s["latency_baseline"] * 1000
    print(
        f"  worker {i}: fast={fast:6.1f}ms  baseline={slow:6.1f}ms  "
        f"drift_factor={s['drift_factor']}  suspended={s['suspended']}"
    )

## 9. Cleanup

In [ ]:
adaptive.stop_health_probes()
for w in workers:
    w.stop()
print("workers stopped")